In [ ]:
# Cell 1: Imports and Data Loading
import pandas as pd
from src import paths

# Import your pipeline functions directly!
from src.barter.pipeline_model_prep import (
    create_market_scope, 
    apply_icf_clustering, 
    filter_na_company_locations,
    start_date,
    filter_low_n_categories,
    filter_incomplete_weeks,
    filter_dead_high_volume_deals,
    filter_archived,
    max_follower_requirement,
    min_apps_after_seven_days,
    min_apps_total,
    filter_invalid_deals,
    filter_partners,
    log_transform_cols
)

# Load the heavy feature dataframe once
df_features = pd.read_parquet(paths.PROCESSED_DATA_DIR / 'BARTER_DEALS_FEATURES.parquet')

# Pre-process the heavy stuff that doesn't change often
df_features['created_at'] = pd.to_datetime(df_features['created_at'])
df_features['month'] = df_features['created_at'].dt.to_period('M').astype(str)
df_features = create_market_scope(df_features)
df_features = apply_icf_clustering(df_features)

In [ ]:
# Cell 2: The Tuning Sandbox
# Tweak these numbers and hit Shift+Enter to instantly see the effect!
test_config = {
    "START_DATE": "2025-04",               # Try changing to 2025-05
    "MIN_CATEGORY_OBSERVATIONS": 30,       # Try changing to 50
    "PARTNER_IDS_TO_REMOVE": [85, 4],
    "diff_total_apps_7_days_floor": 10,
    "MIN_APPS_TOTAL": 0,                   # Try changing to 1
    "min_apps_after_7_days": -1,
    "max_follower_requirement": 25000,
    "LOG_TRANSFORM_COLS": ['apps_after_7_days', 'deal_value', 'online_eligible_last_week']
}

print("Running pipeline with current test config...\n")

clean_df = (
    df_features
    .pipe(filter_na_company_locations)
    .pipe(start_date, start_date=test_config['START_DATE'])
    .pipe(filter_low_n_categories, min_observations=test_config['MIN_CATEGORY_OBSERVATIONS'])
    .pipe(filter_incomplete_weeks)
    .pipe(filter_dead_high_volume_deals, floor=test_config['diff_total_apps_7_days_floor'])
    .pipe(filter_archived)
    .pipe(max_follower_requirement, max_follower_requirement=test_config['max_follower_requirement'])
    .pipe(min_apps_after_seven_days, min_apps_after_7_days=test_config['min_apps_after_7_days'])
    .pipe(min_apps_total, min_apps=test_config['MIN_APPS_TOTAL'])
    .pipe(filter_invalid_deals)
    .pipe(filter_partners, partner_ids_to_remove=test_config['PARTNER_IDS_TO_REMOVE'])
    .pipe(log_transform_cols, cols=test_config['LOG_TRANSFORM_COLS'])
)

print("\nFinal shape:", clean_df.shape)